
# PE07: Language Translation and Detection

## Project Setup and Google Cloud API Configuration

This notebook implements machine translation and language
detection using Google Cloud Translation API (Basic v2).

The implementation uses Python requests to communicate
with Google's official REST API.

### Technologies Used

- Python 3.14.2
- Jupyter Notebook
- Google Cloud Translation API (Basic v2)
- Python requests library
- GitHub Codespaces

### Authentication

The Google Cloud API key is securely stored in GitHub
Codespaces repository secrets and accessed through
Python environment variables.

The API key is not hardcoded in this notebook.

### Initial API Verification

The following cells verify:
1. API authentication.
2. Basic English-to-French translation.
3. Automatic language detection.


In [1]:
import os
import requests

API_KEY = os.getenv("GOOGLE_TRANSLATE_API_KEY")

if not API_KEY:
    raise EnvironmentError(
        "API key not found. Check your GitHub Codespaces secret."
    )

print("Google Cloud API key loaded successfully.")

Google Cloud API key loaded successfully.


In [2]:

BASE_URL = "https://translation.googleapis.com/language/translate/v2"

HEADERS = {
    "x-goog-api-key": API_KEY,
    "Content-Type": "application/json"
}

print("Google Cloud Translation API configured successfully!")


Google Cloud Translation API configured successfully!


In [3]:

payload = {
    "q": "Hello, how are you?",
    "source": "en",
    "target": "fr",
    "format": "text"
}

try:
    response = requests.post(
        BASE_URL,
        headers=HEADERS,
        json=payload,
        timeout=30
    )

    print("HTTP Status Code:", response.status_code)

    if response.status_code == 200:
        result = response.json()

        translated_text = (
            result["data"]["translations"][0]["translatedText"]
        )

        print("Original Text:", payload["q"])
        print("Translated Text:", translated_text)
        print("Source Language: English")
        print("Target Language: French")

    else:
        print("Translation failed.")
        print("Error Details:", response.text)

except requests.exceptions.Timeout:
    print("Error: Request timed out.")

except requests.exceptions.RequestException as error:
    print("Connection Error:", error)


HTTP Status Code: 200
Original Text: Hello, how are you?
Translated Text: Bonjour comment allez-vous?
Source Language: English
Target Language: French


In [4]:

DETECT_URL = BASE_URL + "/detect"

payload = {
    "q": "Bonjour, comment allez-vous?"
}

try:
    response = requests.post(
        DETECT_URL,
        headers=HEADERS,
        json=payload,
        timeout=30
    )

    print("HTTP Status Code:", response.status_code)

    if response.status_code == 200:
        result = response.json()

        detection = result["data"]["detections"][0][0]

        print("Input Text:", payload["q"])
        print("Detected Language:", detection["language"])
        print("Confidence:", detection.get("confidence"))

    else:
        print("Detection failed.")
        print("Error Details:", response.text)

except requests.exceptions.RequestException as error:
    print("Connection Error:", error)


HTTP Status Code: 200
Input Text: Bonjour, comment allez-vous?
Detected Language: fr
Confidence: 1



# PE07: Language Translation and Detection

## Exercise 1: Machine Translation

This exercise implements machine translation using the
Google Cloud Translation API (Basic v2).

Unlike the original exercise instructions, this implementation
uses Google's official REST API through Python requests
instead of the googletrans package.

The implementation includes:
1. Basic text translation.
2. Batch translation.
3. Individual error handling.
4. Support for multiple language pairs.



### Exercise 1 – Part 1: Basic Translation

**Objective:** Translate a single text from a source language
to a destination language using Google Cloud Translation API.

**Function:** translate_text()

**Parameters:**
- text: Text that needs to be translated.
- source_language: Source language code (default: en).
- target_language: Destination language code (default: fr).

**Returns:**
- Translated text as a string.

The function validates the input and handles API-related errors.


In [5]:

def translate_text(text, source_language="en", target_language="fr"):
    """
    Translate a single text using Google Cloud Translation API.

    Parameters:
        text (str): Text to translate.
        source_language (str): Source language code. Default is 'en'.
        target_language (str): Destination language code. Default is 'fr'.

    Returns:
        str: Translated text.

    Raises:
        ValueError: If the input text or language codes are invalid.
        RuntimeError: If the translation API request fails.
    """

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text cannot be empty.")
    
    if not isinstance(source_language, str) or not source_language.strip():
        raise ValueError("Valid source language code is required.")
    
    if not isinstance(target_language, str) or not target_language.strip():
        raise ValueError("Valid target language code is required.")

    payload = {
        "q": text,
        "source": source_language,
        "target": target_language,
        "format": "text"
    }

    try:
        response = requests.post(
            BASE_URL,
            headers=HEADERS,
            json=payload,
            timeout=30
        )

        if response.status_code != 200:
            try:
                error_details = response.json()["error"]
                error_message = error_details.get("message", "Unknown API error"
                )
                
            except (ValueError, KeyError):
                error_message = response.text

            raise RuntimeError(
                f"Translation API error "
                f"({response.status_code}): {error_message}"
            )
        

        result = response.json()

        return result["data"]["translations"][0]["translatedText"]

    except requests.exceptions.RequestException as error:
        raise RuntimeError(f"Translation request failed: {error}")


In [6]:

# Exercise 1 - Part 1: Basic Translation Test

original_text = "Hello, how are you?"

translated_text = translate_text(
    original_text,
    source_language="en",
    target_language="fr"
)

print("Original Text:", original_text)
print("Translated Text:", translated_text)


Original Text: Hello, how are you?
Translated Text: Bonjour comment allez-vous?


In [7]:

# Testing different language combinations

test_cases = [
    ("Good morning!", "en", "fr"),
    ("Hello, how are you?", "en", "es"),
    ("Thank you very much.", "en", "de")
]

for text, source, target in test_cases:

    translation = translate_text(
        text,
        source_language=source,
        target_language=target
    )

    print(f"\nOriginal ({source}): {text}")
    print(f"Translation ({target}): {translation}")



Original (en): Good morning!
Translation (fr): Bonjour!

Original (en): Hello, how are you?
Translation (es): ¿Hola, cómo estás?

Original (en): Thank you very much.
Translation (de): Vielen Dank.



### Exercise 1 – Part 2: Batch Translation Implementation

**Objective:**

Implement a function that translates multiple text inputs
using Google Cloud Translation API.

**Requirements:**

1. Accept a list of texts.
2. Support configurable source and destination language codes.
3. Use English as the default source language.
4. Use French as the default destination language.
5. Return a list of dictionaries.
6. Handle errors individually for each text.

**Expected dictionary fields:**

- original_text
- translated_text
- source_language
- destination_language
- error


In [8]:

def batch_translate(texts, source_language="en", target_language="fr"):
    """
    Translate multiple texts using Google Cloud Translation API.

    Parameters:
        texts (list): List of text strings to translate.
        source_language (str): Source language code. Default is 'en'.
        target_language (str): Target language code. Default is 'fr'.

    Returns:
        list: A list of dictionaries containing original text,
              translated text, source language,
              destination language, and error information.

    Each text is processed independently. An error in one
    translation does not stop the remaining translations.
    """

    if not isinstance(texts, list):
        raise ValueError("Input must be a list of texts.")

    results = []

    for text in texts:

        result = {
            "original_text": text,
            "translated_text": None,
            "source_language": source_language,
            "destination_language": target_language,
            "error": None
        }

        try:
            if not isinstance(text, str) or not text.strip():
                raise ValueError("Text cannot be empty.")

            translated_text = translate_text(
                text,
                source_language=source_language,
                target_language=target_language
            )

            result["translated_text"] = translated_text

        except (ValueError, RuntimeError) as error:
            result["error"] = str(error)

        except Exception as error:
            result["error"] = f"Unexpected error: {error}"

        results.append(result)

    return results


In [9]:

# Exercise 1 - Part 2: Batch Translation Test

sample_texts = [
    "Hello, how are you?",
    "Good morning!",
    "Thank you very much.",
    "What is your name?"
]

batch_results = batch_translate(
    sample_texts,
    source_language="en",
    target_language="fr"
)

for result in batch_results:

    print("\nOriginal Text:", result["original_text"])
    print("Translated Text:", result["translated_text"])
    print("Source Language:", result["source_language"])
    print("Destination Language:", result["destination_language"])
    print("Error:", result["error"])
    print("-" * 60)



Original Text: Hello, how are you?
Translated Text: Bonjour comment allez-vous?
Source Language: en
Destination Language: fr
Error: None
------------------------------------------------------------

Original Text: Good morning!
Translated Text: Bonjour!
Source Language: en
Destination Language: fr
Error: None
------------------------------------------------------------

Original Text: Thank you very much.
Translated Text: Merci beaucoup.
Source Language: en
Destination Language: fr
Error: None
------------------------------------------------------------

Original Text: What is your name?
Translated Text: Quel est ton nom?
Source Language: en
Destination Language: fr
Error: None
------------------------------------------------------------


In [10]:

# Testing individual error handling

error_test_texts = [
    "Good morning!",
    "",
    "   ",
    "Thank you for your help!"
]

error_results = batch_translate(
    error_test_texts,
    source_language="en",
    target_language="fr"
)

for result in error_results:

    print("\nOriginal:", repr(result["original_text"]))

    if result["error"]:
        print("Status: Failed")
        print("Error:", result["error"])
    else:
        print("Status: Successful")
        print("Translation:", result["translated_text"])



Original: 'Good morning!'
Status: Successful
Translation: Bonjour!

Original: ''
Status: Failed
Error: Text cannot be empty.

Original: '   '
Status: Failed
Error: Text cannot be empty.

Original: 'Thank you for your help!'
Status: Successful
Translation: Merci pour votre aide!


In [11]:

# Testing invalid language code

invalid_language_result = batch_translate(
    ["Hello, how are you?"],
    source_language="invalid",
    target_language="fr"
)

print(invalid_language_result)


[{'original_text': 'Hello, how are you?', 'translated_text': None, 'source_language': 'invalid', 'destination_language': 'fr', 'error': 'Translation API error (400): Invalid Value'}]



# Exercise 2: Language Detection

This exercise implements language detection using the
Google Cloud Translation API (Basic v2).

The system identifies the source language of input text
and provides the detected language code and language name.

The implementation includes:

1. Basic language detection.
2. Language code identification.
3. Language name mapping.
4. Error handling and input validation.


In [12]:

# Exercise 2 - Part 1: Basic Language Detection

DETECT_URL = BASE_URL + "/detect"

print("Language Detection API configured successfully!")


Language Detection API configured successfully!


In [13]:

def detect_language(text):
    """
    Detect the language of an input text using
    Google Cloud Translation API.

    Parameters:
        text (str): Text whose language needs to be detected.

    Returns:
        dict: Contains detected language code and confidence.

    Raises:
        ValueError: If the input text is empty or invalid.
        RuntimeError: If the API request fails.
    """

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text cannot be empty.")

    payload = {
        "q": text
    }

    try:
        response = requests.post(
            DETECT_URL,
            headers=HEADERS,
            json=payload,
            timeout=30
        )

        if response.status_code != 200:
            try:
                error_details = response.json()["error"]
                error_message = error_details.get(
                    "message", "Unknown API error"
                )
            except (ValueError, KeyError):
                error_message = response.text

            raise RuntimeError(
                f"Language detection API error "
                f"({response.status_code}): {error_message}"
            )

        result = response.json()

        detection = result["data"]["detections"][0][0]

        return {
            "language": detection["language"],
            "confidence": detection.get("confidence")
        }

    except requests.exceptions.RequestException as error:
        raise RuntimeError(
            f"Language detection request failed: {error}"
        )


In [14]:

def get_language_names():
    """
    Retrieve supported language codes and their English names
    from Google Cloud Translation API.

    Returns:
        dict: Language codes mapped to language names.
    """

    languages_url = BASE_URL + "/languages"

    try:
        response = requests.get(
            languages_url,
            headers=HEADERS,
            params={"target": "en"},
            timeout=30
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"Language list API error: {response.text}"
            )

        result = response.json()

        return {
            item["language"]: item["name"]
            for item in result["data"]["languages"]
        }

    except requests.exceptions.RequestException as error:
        raise RuntimeError(
            f"Unable to retrieve language names: {error}"
        )


# Load supported language names
LANGUAGE_NAMES = get_language_names()

print("Supported language names loaded:", len(LANGUAGE_NAMES))


Supported language names loaded: 195


In [15]:

# Basic Language Detection Test

sample_text = "Guten Tag, wie geht es dir?"

detection_result = detect_language(sample_text)

language_code = detection_result["language"]

language_name = LANGUAGE_NAMES.get(
    language_code, "Unknown"
)

print("Input Text:", sample_text)

print(
    f"Detected language: {language_code} "
    f"({language_name})"
)

print(
    "Confidence:",
    detection_result["confidence"]
)


Input Text: Guten Tag, wie geht es dir?
Detected language: de (German)
Confidence: 1


In [16]:

# Testing language detection with multiple languages

sample_texts = [
    "Hello, how are you?",
    "Bonjour, comment allez-vous?",
    "Hola, ¿cómo estás?",
    "Guten Tag, wie geht es dir?"
]

for text in sample_texts:

    result = detect_language(text)

    code = result["language"]

    name = LANGUAGE_NAMES.get(code, "Unknown")

    print("\nInput Text:", text)
    print(f"Detected Language: {code} ({name})")
    print("Confidence:", result["confidence"])
    print("-" * 50)



Input Text: Hello, how are you?
Detected Language: en (English)
Confidence: 0.989586591720581
--------------------------------------------------

Input Text: Bonjour, comment allez-vous?
Detected Language: fr (French)
Confidence: 1
--------------------------------------------------

Input Text: Hola, ¿cómo estás?
Detected Language: es (Spanish)
Confidence: 1
--------------------------------------------------

Input Text: Guten Tag, wie geht es dir?
Detected Language: de (German)
Confidence: 1
--------------------------------------------------


In [17]:

# Testing empty input validation

try:
    detect_language("")

except ValueError as error:
    print("Validation Error:", error)


Validation Error: Input text cannot be empty.


In [18]:
try:
    detect_language("   ")

except ValueError as error:
    print("Validation Error:", error)

Validation Error: Input text cannot be empty.



## Exercise 2 – Part 2: Integrated Translation System

### Objective

Develop a class-based language-processing system using
Google Cloud Translation API (Basic v2).

The system integrates language detection and machine
translation into a single reusable Python class.

### System Features

1. Automatic source-language detection.
2. Single-text translation.
3. Multiple-text processing.
4. Translation into multiple target languages.
5. Language code and language name mapping.
6. Input validation.
7. Comprehensive API error handling.

### Class Design

Class Name: LanguageProcessingSystem

Methods:
- detect_language()
- translate_text()
- process_multiple_texts()
- translate_multiple_targets()
- get_language_name()


In [19]:

class LanguageProcessingSystem:
    """
    Integrated language detection and translation system
    using Google Cloud Translation API (Basic v2).
    """

    def __init__(self, api_key, base_url, language_names):
        """
        Initialize the language-processing system.

        Parameters:
            api_key (str): Google Cloud API key.
            base_url (str): Google Translation API endpoint.
            language_names (dict): Language codes mapped to names.
        """

        if not api_key:
            raise ValueError("Google Cloud API key is required.")

        self.api_key = api_key
        self.base_url = base_url
        self.language_names = language_names

        self.headers = {
            "x-goog-api-key": self.api_key,
            "Content-Type": "application/json"
        }

        self.detect_url = self.base_url + "/detect"

    def get_language_name(self, language_code):
        """
        Return the English name of a language.

        Parameters:
            language_code (str): Language code such as 'en' or 'fr'.

        Returns:
            str: Language name or 'Unknown'.
        """

        if not isinstance(language_code, str) or not language_code.strip():
            return "Unknown"

        return self.language_names.get(
            language_code.lower(), "Unknown"
        )

    def _handle_response(self, response, operation):
        """
        Validate an API response and return its JSON content.
        """

        if response.status_code != 200:

            try:
                error_details = response.json()["error"]
                message = error_details.get(
                    "message", "Unknown API error"
                )

            except (ValueError, KeyError, TypeError):
                message = response.text

            raise RuntimeError(
                f"{operation} failed "
                f"({response.status_code}): {message}"
            )

        try:
            return response.json()

        except ValueError:
            raise RuntimeError(
                f"{operation} returned invalid JSON."
            )

    def detect_language(self, text):
        """
        Automatically detect the language of input text.

        Parameters:
            text (str): Text to analyze.

        Returns:
            dict: Detected language code, language name,
                  and confidence score.
        """

        if not isinstance(text, str) or not text.strip():
            raise ValueError("Input text cannot be empty.")

        payload = {"q": text}

        try:
            response = requests.post(
                self.detect_url,
                headers=self.headers,
                json=payload,
                timeout=30
            )

            result = self._handle_response(
                response, "Language detection"
            )

            detection = result["data"]["detections"][0][0]

            code = detection["language"]

            return {
                "language_code": code,
                "language_name": self.get_language_name(code),
                "confidence": detection.get("confidence")
            }

        except requests.exceptions.RequestException as error:
            raise RuntimeError(
                f"Language detection request failed: {error}"
            )

        except (KeyError, IndexError, TypeError):
            raise RuntimeError(
                "Unexpected language detection response."
            )

    def translate_text(
        self,
        text,
        target_language="fr",
        source_language=None
    ):
        """
        Translate a single text.

        If source_language is None, the source language
        is detected automatically.

        Parameters:
            text (str): Text to translate.
            target_language (str): Destination language code.
            source_language (str or None): Source language code.
                None enables automatic detection.

        Returns:
            dict: Original text, translated text, source language,
                  destination language, and confidence information.
        """

        if not isinstance(text, str) or not text.strip():
            raise ValueError("Input text cannot be empty.")

        if not isinstance(target_language, str) or not target_language.strip():
            raise ValueError("Valid target language code is required.")

        if source_language is not None:
            if not isinstance(source_language, str) or not source_language.strip():
                raise ValueError("Valid source language code is required.")

        detection = None

        # Automatically detect the source language when not provided
        if source_language is None:
            detection = self.detect_language(text)
            source_language = detection["language_code"]

        # FIX: Handle identical source and target languages
        # without making an unnecessary API translation request.
        if source_language.lower() == target_language.lower():

            return {
                "original_text": text,
                "translated_text": text,
                "source_language": source_language,
                "source_language_name": self.get_language_name(
                    source_language
                ),
                "target_language": target_language,
                "target_language_name": self.get_language_name(
                    target_language
                ),
                "confidence": (
                    detection["confidence"] if detection else None
                ),
                "error": None
            }

        payload = {
            "q": text,
            "source": source_language,
            "target": target_language,
            "format": "text"
        }

        try:
            response = requests.post(
                self.base_url,
                headers=self.headers,
                json=payload,
                timeout=30
            )

            result = self._handle_response(
                response, "Translation"
            )

            translated_text = (
                result["data"]["translations"][0]["translatedText"]
            )

            return {
                "original_text": text,
                "translated_text": translated_text,
                "source_language": source_language,
                "source_language_name": self.get_language_name(
                    source_language
                ),
                "target_language": target_language,
                "target_language_name": self.get_language_name(
                    target_language
                ),
                "confidence": (
                    detection["confidence"] if detection else None
                ),
                "error": None
            }

        except requests.exceptions.RequestException as error:
            raise RuntimeError(
                f"Translation request failed: {error}"
            )

        except (KeyError, IndexError, TypeError):
            raise RuntimeError(
                "Unexpected translation API response."
            )

    def process_multiple_texts(
        self,
        texts,
        target_language="fr",
        source_language=None
    ):
        """
        Process multiple texts independently.

        Automatically detects each source language when
        source_language is None.

        Parameters:
            texts (list): List of text strings.
            target_language (str): Destination language code.
            source_language (str or None): Source language code.

        Returns:
            list: Translation result dictionaries.
        """

        if not isinstance(texts, list):
            raise ValueError("Input must be a list of texts.")

        results = []

        for text in texts:

            try:
                result = self.translate_text(
                    text,
                    target_language=target_language,
                    source_language=source_language
                )

            except (ValueError, RuntimeError) as error:

                result = {
                    "original_text": text,
                    "translated_text": None,
                    "source_language": None,
                    "source_language_name": None,
                    "target_language": target_language,
                    "target_language_name": self.get_language_name(
                        target_language
                    ),
                    "confidence": None,
                    "error": str(error)
                }

            except Exception as error:

                result = {
                    "original_text": text,
                    "translated_text": None,
                    "source_language": None,
                    "source_language_name": None,
                    "target_language": target_language,
                    "target_language_name": self.get_language_name(
                        target_language
                    ),
                    "confidence": None,
                    "error": f"Unexpected error: {error}"
                }

            results.append(result)

        return results

    def translate_multiple_targets(
        self,
        text,
        target_languages,
        source_language=None
    ):
        """
        Translate one text into multiple target languages.

        Parameters:
            text (str): Original input text.
            target_languages (list): Destination language codes.
            source_language (str or None): Source language code.

        Returns:
            list: Translation results for each target language.
        """

        if not isinstance(text, str) or not text.strip():
            raise ValueError("Input text cannot be empty.")

        if not isinstance(target_languages, list) or not target_languages:
            raise ValueError("Provide at least one target language.")

        if source_language is None:
            detection = self.detect_language(text)
            source_language = detection["language_code"]

        results = []

        for target in target_languages:

            try:
                result = self.translate_text(
                    text,
                    target_language=target,
                    source_language=source_language
                )

            except (ValueError, RuntimeError) as error:

                result = {
                    "original_text": text,
                    "translated_text": None,
                    "source_language": source_language,
                    "target_language": target,
                    "error": str(error)
                }

            results.append(result)

        return results

    def __repr__(self):
        return "LanguageProcessingSystem(Google Cloud Translation API)"


In [20]:

# Initialize the integrated language-processing system

language_system = LanguageProcessingSystem(
    api_key=API_KEY,
    base_url=BASE_URL,
    language_names=LANGUAGE_NAMES
)

print("Integrated Language Processing System initialized!")
print(language_system)


Integrated Language Processing System initialized!
LanguageProcessingSystem(Google Cloud Translation API)


In [21]:

# Test automatic language detection

text = "Guten Tag, wie geht es dir?"

detection = language_system.detect_language(text)

print("Input Text:", text)
print("Detected Code:", detection["language_code"])
print("Detected Language:", detection["language_name"])
print("Confidence:", detection["confidence"])


Input Text: Guten Tag, wie geht es dir?
Detected Code: de
Detected Language: German
Confidence: 1


In [22]:

# Test single-text translation with automatic detection

result = language_system.translate_text(
    "Guten Tag, wie geht es dir?",
    target_language="fr"
)

print("Original Text:", result["original_text"])
print("Source Language:", result["source_language_name"])
print("Translated Text:", result["translated_text"])
print("Target Language:", result["target_language_name"])
print("Confidence:", result["confidence"])


Original Text: Guten Tag, wie geht es dir?
Source Language: German
Translated Text: Bonjour, comment allez-vous ?
Target Language: French
Confidence: 1


In [23]:

# Test multiple texts with automatic source-language detection

sample_texts = [
    "Hello, how are you?",
    "Bonjour, comment allez-vous?",
    "Hola, ¿cómo estás?",
    "Guten Tag, wie geht es dir?"
]

results = language_system.process_multiple_texts(
    sample_texts,
    target_language="en"
)

for item in results:

    print("\nOriginal Text:", item["original_text"])

    print(
        "Detected Source:",
        item["source_language_name"],
        f"({item['source_language']})"
    )

    print("English Translation:", item["translated_text"])

    print("Error:", item["error"])

    print("-" * 60)



Original Text: Hello, how are you?
Detected Source: English (en)
English Translation: Hello, how are you?
Error: None
------------------------------------------------------------

Original Text: Bonjour, comment allez-vous?
Detected Source: French (fr)
English Translation: Good morning, how are you doing?
Error: None
------------------------------------------------------------

Original Text: Hola, ¿cómo estás?
Detected Source: Spanish (es)
English Translation: Hello how are you?
Error: None
------------------------------------------------------------

Original Text: Guten Tag, wie geht es dir?
Detected Source: German (de)
English Translation: Good day how are you?
Error: None
------------------------------------------------------------


In [24]:

# Test translation into multiple target languages

original_text = "Welcome to our language translation system."

target_languages = ["fr", "es", "de"]

multi_target_results = language_system.translate_multiple_targets(
    original_text,
    target_languages,
    source_language="en"
)

for item in multi_target_results:

    print("\nOriginal Text:", item["original_text"])

    print(
        "Target Language:",
        item["target_language_name"]
    )

    print(
        "Translated Text:",
        item["translated_text"]
    )

    print("-" * 50)



Original Text: Welcome to our language translation system.
Target Language: French
Translated Text: Bienvenue sur notre système de traduction.
--------------------------------------------------

Original Text: Welcome to our language translation system.
Target Language: Spanish
Translated Text: Bienvenido a nuestro sistema de traducción de idiomas.
--------------------------------------------------

Original Text: Welcome to our language translation system.
Target Language: German
Translated Text: Willkommen bei unserem Sprachübersetzungssystem.
--------------------------------------------------


In [25]:

# Test error handling in the integrated system

error_test_cases = [
    "Hello, how are you?",
    "",
    "   ",
    "Thank you for your support."
]

error_results = language_system.process_multiple_texts(
    error_test_cases,
    target_language="fr"
)

for item in error_results:

    print("\nInput:", repr(item["original_text"]))

    if item["error"]:
        print("Status: Failed")
        print("Error:", item["error"])
    else:
        print("Status: Successful")
        print("Translation:", item["translated_text"])



Input: 'Hello, how are you?'
Status: Successful
Translation: Bonjour comment allez-vous?

Input: ''
Status: Failed
Error: Input text cannot be empty.

Input: '   '
Status: Failed
Error: Input text cannot be empty.

Input: 'Thank you for your support.'
Status: Successful
Translation: Merci pour votre soutien.


In [26]:

# Test language name mapping

language_codes = ["en", "fr", "es", "de", "ja"]

for code in language_codes:

    name = language_system.get_language_name(code)

    print(f"{code} -> {name}")


en -> English
fr -> French
es -> Spanish
de -> German
ja -> Japanese



# Comprehensive Testing and Validation

This section evaluates the reliability of the integrated
language-processing system.

The following test cases are performed:

1. Translation between different language pairs.
2. Automatic detection of multiple source languages.
3. Empty and whitespace-only input.
4. Invalid language codes.
5. Very long text containing more than 1,000 characters.
6. API failures and timeout handling.

The objective is to verify input validation, API communication,
error handling, and the overall reliability of the system.


In [27]:

# Test 1: Invalid language codes

invalid_language_tests = [
    {
        "name": "Invalid source language",
        "text": "Hello, how are you?",
        "source": "invalid",
        "target": "fr"
    },
    {
        "name": "Invalid target language",
        "text": "Hello, how are you?",
        "source": "en",
        "target": "invalid"
    }
]

for test in invalid_language_tests:

    print("\nTest:", test["name"])

    try:
        result = language_system.translate_text(
            test["text"],
            source_language=test["source"],
            target_language=test["target"]
        )

        print("Unexpected success:", result["translated_text"])

    except (ValueError, RuntimeError) as error:
        print("Expected error handled successfully.")
        print("Error:", error)

    print("-" * 60)



Test: Invalid source language
Expected error handled successfully.
Error: Translation failed (400): Invalid Value
------------------------------------------------------------

Test: Invalid target language
Expected error handled successfully.
Error: Translation failed (400): Invalid Value
------------------------------------------------------------


In [28]:

# Test 2: Empty and invalid input validation

invalid_inputs = [
    "",
    "   ",
    None,
    123
]

for test_input in invalid_inputs:

    print("\nTesting input:", repr(test_input))

    try:
        result = language_system.translate_text(
            test_input,
            target_language="fr"
        )

        print("Unexpected success:", result)

    except ValueError as error:
        print("Validation successful:", error)

    except Exception as error:
        print("Unexpected error:", error)



Testing input: ''
Validation successful: Input text cannot be empty.

Testing input: '   '
Validation successful: Input text cannot be empty.

Testing input: None
Validation successful: Input text cannot be empty.

Testing input: 123
Validation successful: Input text cannot be empty.


In [29]:

# Test 3: Very long text translation

long_text = (
    "Machine translation is an important application of "
    "artificial intelligence and natural language processing. "
    "It allows people who speak different languages to "
    "communicate more effectively. Language detection helps "
    "identify the original language of a given text, while "
    "translation systems convert the text into another language. "
    "Modern translation services use advanced machine learning "
    "models to understand sentences and generate meaningful "
    "translations. These technologies are useful in education, "
    "business, healthcare, tourism, and international communication. "
) * 3

print("Original text length:", len(long_text))
print("Meets 1000-character requirement:", len(long_text) >= 1000)

try:
    long_text_result = language_system.translate_text(
        long_text,
        source_language="en",
        target_language="fr"
    )

    print("\nTranslation successful!")
    print("Source Language:", long_text_result["source_language_name"])
    print("Target Language:", long_text_result["target_language_name"])
    print("Translated Text Length:", len(long_text_result["translated_text"]))
    print("\nTranslated Text:")
    print(long_text_result["translated_text"])

except (ValueError, RuntimeError) as error:
    print("Long text translation failed:", error)


Original text length: 1686
Meets 1000-character requirement: True

Translation successful!
Source Language: English
Target Language: French
Translated Text Length: 675

Translated Text:
La traduction automatique est une application importante de l'intelligence artificielle et du traitement automatique du langage naturel. Elle permet aux personnes parlant des langues différentes de communiquer plus efficacement. La détection de la langue aide à identifier la langue originale d'un texte donné, tandis que les systèmes de traduction convertissent le texte dans une autre langue. Les services de traduction modernes utilisent des modèles d'apprentissage automatique avancés pour comprendre les phrases et générer des traductions pertinentes. Ces technologies sont utiles dans l'éducation, les affaires, la santé, le tourisme et la communication internationale.


In [30]:

# Test 4A: Simulate API timeout

from unittest.mock import patch

try:
    with patch(
        "requests.post",
        side_effect=requests.exceptions.Timeout(
            "Simulated API timeout"
        )
    ):
        language_system.translate_text(
            "Hello, how are you?",
            source_language="en",
            target_language="fr"
        )

except RuntimeError as error:
    print("API timeout handled successfully.")
    print("Error:", error)


API timeout handled successfully.
Error: Translation request failed: Simulated API timeout


In [31]:

# Test 4B: Simulate API permission failure

from unittest.mock import Mock

mock_response = Mock()

mock_response.status_code = 403

mock_response.json.return_value = {
    "error": {
        "code": 403,
        "message": "Simulated API permission error"
    }
}

try:
    with patch(
        "requests.post",
        return_value=mock_response
    ):
        language_system.translate_text(
            "Hello, how are you?",
            source_language="en",
            target_language="fr"
        )

except RuntimeError as error:
    print("API permission error handled successfully.")
    print("Error:", error)


API permission error handled successfully.
Error: Translation failed (403): Simulated API permission error


In [32]:

# Test 5: Complete multilingual processing workflow

multilingual_inputs = [
    "Hello, how are you?",
    "Bonjour, comment allez-vous?",
    "Hola, ¿cómo estás?",
    "Guten Tag, wie geht es dir?"
]

final_results = language_system.process_multiple_texts(
    multilingual_inputs,
    target_language="en"
)

successful = 0
failed = 0

for result in final_results:

    print("\nOriginal Text:", result["original_text"])

    print(
        "Detected Language:",
        result["source_language_name"],
        f"({result['source_language']})"
    )

    if result["error"] is None:
        print("English Translation:", result["translated_text"])
        successful += 1
    else:
        print("Error:", result["error"])
        failed += 1

    print("-" * 60)

print("\nFINAL TEST SUMMARY")
print("Total Inputs:", len(final_results))
print("Successful:", successful)
print("Failed:", failed)



Original Text: Hello, how are you?
Detected Language: English (en)
English Translation: Hello, how are you?
------------------------------------------------------------

Original Text: Bonjour, comment allez-vous?
Detected Language: French (fr)
English Translation: Good morning, how are you doing?
------------------------------------------------------------

Original Text: Hola, ¿cómo estás?
Detected Language: Spanish (es)
English Translation: Hello how are you?
------------------------------------------------------------

Original Text: Guten Tag, wie geht es dir?
Detected Language: German (de)
English Translation: Good day how are you?
------------------------------------------------------------

FINAL TEST SUMMARY
Total Inputs: 4
Successful: 4
Failed: 0



# Conclusion

## Project Summary

This project successfully implements language translation
and language detection using the Google Cloud Translation
API (Basic v2).

The implementation includes individual text translation,
batch translation, automatic language detection, multilingual
processing, and translation into multiple target languages.

A reusable Python class, LanguageProcessingSystem, integrates
these features into a single system.

## Error Handling and Validation

The system validates text inputs and language codes and handles
API-related failures, including invalid language codes,
request timeouts, and permission errors.

The implementation was tested using empty inputs, invalid
inputs, long text, multiple languages, and simulated API failures.

## Assumptions

1. A valid Google Cloud Translation API key is available.
2. The Translation API is enabled in the Google Cloud project.
3. The API key has the required permissions.
4. The source and target languages are supported by the API.

## Limitations

1. Translation quality may vary depending on the language pair
   and context.
2. Language detection may be less reliable for very short text.
3. API usage depends on Google Cloud quotas, billing, and
   service availability.
4. Internet connectivity is required for API requests.

## Final Result

The project successfully demonstrates machine translation,
language detection, batch processing, multilingual support,
and API error handling using Python and Google Cloud.
